# Fraunhofer Diffraction Simulator

Interactive 2D simulator for **slit**, **rectangular**, **circular**, coherent **multiple/mixed**, and arbitrary **custom** apertures (Taller 4, exercise 20).

The implementation follows the spatial Fourier-transform relations in [`theory and context/Fraunhofer_Transfomada_Fourier_espacial.pdf`](theory%20and%20context/Fraunhofer_Transfomada_Fourier_espacial.pdf):

- Rectangle/slit: $A(f_x,f_y)=ab\,\mathrm{sinc}(af_x)\,\mathrm{sinc}(bf_y)$.
- Circle: $A(f_r)=\pi r^2\,2J_1(2\pi r f_r)/(2\pi r f_r)$.
- Translated coherent openings: amplitudes acquire $e^{-i2\pi(f_xx_0+f_yy_0)}$ and are summed before taking $|A|^2$.
- Custom expression/image: $I\propto|\mathcal{F}\{t(\tilde{x},\tilde{y})\}|^2$ is evaluated by a centered, scaled 2D FFT.
- Screen mapping: $f_x=x'/(\lambda z)$ and $f_y=y'/(\lambda z)$, where $\lambda=\lambda_0/n$.

Before a field or FFT is evaluated, the simulator enforces $N_F=R_{\max}^2/(\lambda z)\le N_{F,\max}$. The default $N_{F,\max}=0.1$ is an explicit interpretation of the source's far-field requirement; it can be made stricter in **Advanced controls**.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

import dashboard_tools as dashboard
import diffraction_config as config
import diffraction_engine as engine
import fourier_transform as fourier
import screen_tools as screen

print("Python:", sys.executable)
for module in (np, plt.matplotlib, widgets):
    print(f"  {module.__name__}: {module.__version__}")
print("Analytical and numerical diffraction tools loaded OK.")

## 1. Analytical sanity checks

These checks validate the implementation before the interactive UI is constructed:

1. a rectangular/slit transform vanishes at $f_x=1/a$;
2. the circular transform vanishes at the first zero of $J_1$;
3. changing refractive index changes the in-medium wavelength as $\lambda_0/n$;
4. an invalid near-field configuration is rejected by the mandatory gate.

In [ ]:
from scipy.special import jn_zeros

lambda_0 = config.WAVELENGTH_NM.to_si(config.WAVELENGTH_NM.default)
slit = engine.Aperture.slit(
    config.DEFAULT_SLIT_WIDTH_M,
    config.DEFAULT_SLIT_LENGTH_M,
)
circle = engine.Aperture.circle(config.DEFAULT_CIRCLE_RADIUS_M)

# Rectangle/slit first transform zero.
fx = np.array([0.0, 1.0 / slit.width])
slit_field = engine.aperture_amplitude(fx, np.zeros_like(fx), slit)
np.testing.assert_allclose(slit_field[0], slit.area)
np.testing.assert_allclose(slit_field[1], 0.0, atol=slit.area * 1e-12)

# Circular first transform zero: 2*pi*r*fr is the first root of J1.
first_j1_root = jn_zeros(1, 1)[0]
fr_zero = first_j1_root / (2.0 * np.pi * circle.radius)
circle_zero = engine.aperture_amplitude(fr_zero, 0.0, circle)
np.testing.assert_allclose(circle_zero, 0.0, atol=circle.area * 1e-12)

# Wavelength in a medium.
np.testing.assert_allclose(
    engine.medium_wavelength(lambda_0, config.REFRACTIVE_INDEX.maximum),
    lambda_0 / config.REFRACTIVE_INDEX.maximum,
)

# The gate must reject a distance below the reported minimum.
probe = engine.evaluate_far_field(
    [slit], lambda_0, config.DISTANCE_M.default,
    max_fresnel_number=config.MAX_FRESNEL_NUMBER.default,
)
try:
    engine.require_far_field(
        [slit], lambda_0, probe.required_distance / 2.0,
        max_fresnel_number=config.MAX_FRESNEL_NUMBER.default,
    )
except engine.FarFieldError:
    pass
else:
    raise AssertionError("The far-field gate accepted an invalid configuration.")

print("All analytical sanity checks passed.")

## 2. Interactive dashboard

The simulator behaves like a compact optical workbench rather than a graph-only notebook:

1. The top schematic shows the source, aperture plane, propagation distance, and observation plane.
2. The center dashboard keeps the controls, aperture shape, and wavelength-colored diffraction camera visible together.
3. Choose **Slit**, **Rectangle**, **Circle**, **Multiple**, or **Custom** with the case buttons. Mixed mode supports independently sized, positioned, and oriented analytical openings.
4. In **Custom**, choose an expression or image. Expressions use `x`, `y`, and `r` in millimetres and only the listed safe operators/functions. The **Invert white / black** toggle complements either an expression or image mask; set the physical width and height before transforming.
5. Mask resolution controls aperture-edge sampling, while FFT padding controls frequency-sample density. Neither changes the physical aperture dimensions.
6. **Auto update** refreshes after controls change; use **Refresh** at any time or **Reset** to restore defaults. Optional **Profiles** appear below the visual dashboard.

The aperture and optical schematic are always shown. If the far-field check fails, the program does not calculate Fraunhofer diffraction; the observation panel instead reports the minimum valid distance while leaving the selected optical setup visible.

## Dashboard implementation

The complete widget layout, callbacks, aperture builders, and rendering logic live in [diffraction_style.py](diffraction_style.py). The next cell only creates and displays the dashboard.

In [ ]:
import diffraction_style

fraunhofer_dashboard = diffraction_style.FraunhoferDashboard()
fraunhofer_dashboard.show()

### Suggested experiments

- Increase slit width and verify that the horizontal central maximum becomes narrower ($x_{\min}=\lambda z/b$).
- Increase refractive index while holding vacuum wavelength fixed; the pattern contracts because $\lambda=\lambda_0/n$.
- Reduce distance until the validity gate blocks the calculation, then use the reported minimum distance.
- In mixed mode, start with two equal slits and vary their center separation to change the interference-fringe spacing.
- Change one mixed-mode row to **Circle** or **Rectangle** to observe coherent interference between different analytical aperture shapes.
- In Custom expression mode, try `(r <= 0.18) & ~(r < 0.12)` for an annular opening, then increase FFT padding to compare frequency sampling.
- Upload a simple black-and-white logo, calibrate its physical dimensions, and compare white=open with the inverted mask.